In [1]:
import sys, os
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

import os
print("ENV            :", os.getenv("ENV"))
print("LLM_MODEL      :", os.getenv("LLM_MODEL"))
print("REDIS_URL      :", os.getenv("REDIS_URL"))
print("VECTOR DIR     :", os.getenv("LOCAL_VECTOR_DIR"))
print("PRODUCTS_SOURCE:", os.getenv("PRODUCTS_SOURCE"))

from app.settings import settings
print("Settings loaded for env:", settings.env)


python-dotenv could not parse statement starting at line 25


ENV            : dev
LLM_MODEL      : llama-3.3-70b-versatile
REDIS_URL      : redis://localhost:6379/0
VECTOR DIR     : .local_vector_store
PRODUCTS_SOURCE: mcp
Settings loaded for env: dev


In [2]:
from app.rag.loaders import load_text_folder
from app.rag.retriever import build_or_load_index
docs = load_text_folder("data/raw")
print(f"Found {len(docs)} raw documents.")
_ = build_or_load_index(docs if len(docs) > 0 else None)
print("Vector index is ready (built or loaded).")

print(_)

Found 1 raw documents.


/Users/saurabh/Desktop/NN/chatbot/rag_chatbot/app/rag/retriever.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _EMB = HuggingFaceEmbeddings(model_name=settings.embeddings_model)


Vector index is ready (built or loaded).


In [3]:
import redis
from app.settings import settings

r = redis.from_url(settings.redis_url, decode_responses=True)
print("PING →", r.ping())


PING → True


In [4]:
from pprint import pprint

def show(resp):
    print("session_id     :", resp.get("session_id"))
    print("intent         :", resp.get("intent"))
    print("used_rag       :", resp.get("used_rag"))
    print("rag_best_score :", resp.get("rag_best_score"))
    print("redaction      :", resp.get("redaction"))
    print("scope          :", resp.get("scope"))
    print("escalation?    :", "YES" if resp.get("escalation_banner") else "no")
    print("\n--- ANSWER ---\n")
    print(resp.get("answer", "")[:1200])
    print("\n--- PRODUCTS ---")
    for p in resp.get("products", []):
        print("-", p.get("name"))
    print("\n--- MEMORY (recent) ---")
    for t in resp.get("memory_turns", []):
        print(f"{t['role'].upper()}: {t['content'][:160]}")


In [ ]:
from app.chains.chat_pipeline import chat_once

# res1 = chat_once(
#     "Who is rahul dravid?",
#     session_id=None
# )

# res1 = chat_once(
#     "What are some tips to recover faster after childbirth?",
#     session_id=None
# )

res1 = await chat_once(
    "Which breast pump is better for daily use as a new mother?",
    session_id=None
)

print("\n\n=== CHAT RESPONSE ===\n",res1)

show(res1)

sid = res1["session_id"]
sid





=== CHAT RESPONSE ===
 <coroutine object chat_once at 0x306b63bd0>


AttributeError: 'coroutine' object has no attribute 'get'

In [ ]:
# res1 = chat_once(
#     "My baby has a high fever. Should I give paracetamol?",
#     session_id=None
# )
# show(res1)


In [ ]:
# res1 = chat_once(
#     "Can you suggest a breast pump brand that is not in your product list?",
#     session_id=None
# )
# show(res1)

In [ ]:
res1 = chat_once(
    "My name is Riya, but don’t use it. Just answer normally,What are some tips to recover faster after childbirth?",
    session_id=None
)